In [53]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../../utils/")
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from training_models.classification_models import ClassificationModels
from joblib import dump
import json
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [54]:
def undersampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para submuestrear
    undersampler = RandomUnderSampler(sampling_strategy='not minority', random_state=seed)

    #Se aplica el submuestreo
    X_res, y_res= undersampler.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [55]:
def oversampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para sobremuestrear
    smote = SMOTE(random_state=seed)

    #Se aplica el sobremuestreo
    X_res, y_res= smote.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [56]:
def split(df_data, seed):
    #Separa los datos
    data_under= undersampling(df_data, seed)
    data_over= oversampling(df_data, seed)
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    train_data_under, val_data_under = train_test_split(data_under, test_size=0.2, random_state=seed)
    train_data_over, val_data_over = train_test_split(data_over, test_size=0.2, random_state=seed)
    #print(train_data.shape, val_data.shape, train_data_under.shape, val_data_under.shape, train_data_over.shape, val_data_over.shape)
    #print(train_data['target'].value_counts(), val_data['target'].value_counts(), train_data_under['target'].value_counts(), val_data_under['target'].value_counts(), train_data_over['target'].value_counts(), val_data_over['target'].value_counts())
    return train_data, val_data, train_data_under, val_data_under, train_data_over, val_data_over

In [ ]:
def train(train_v, validation_v, iteration, seed):
    X_train = train_v.drop(columns="target").values
    y_train = train_v["target"].values
    X_val = validation_v.drop(columns="target").values
    y_val = validation_v["target"].values

    print(f"GridSearchCV for Random Forest, iteration: {iteration}")

    param_grid = {
        "n_estimators": [100],
        "criterion": ["gini"],
        "min_samples_split": [2],
        "min_samples_leaf": [1],
        "max_features": ["sqrt"],
        "max_depth": [10]
    }

    rf = RandomForestClassifier(random_state=seed)
    rf.fit(X_train, y_train)

    dump(rf, f"../../models/RandomForest_Grid_{seed}.joblib")
    #sacar metricas de rendimiento del modelo inicial
    #agregar validacion cruzada

    grid = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring="f1_weighted", n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_params = grid.best_params_

    print(f"Best GridSearch params iteration {iteration}: {best_params}")
    dump(best_model, f"../../models/RandomForest_Grid_{seed}_best.joblib")


In [59]:
rename_map = {
    "f1_weighted": "F1-score",
    "recall_weighted": "Recall",
    "precision_weighted": "Precision",
    "accuracy": "Accuracy"
}

In [60]:
def metrics(perf, iteration, seed, sampling):
    #Se obtienen las metricas de entrenamiento y validacion en variables diferentes
    train_metrics = perf["training_metrics"]
    val_metrics = perf["validation_metrics"]
    print(val_metrics)
    #Se elimina la matriz de confusiones
    val_metrics= val_metrics.copy()
    val_metrics.pop("Confusion Matrix", None)
    #Renombra metricas
    train_renamed = {rename_map.get(k, k): v for k, v in train_metrics.items()}
    #Se asignan los valores de las metricas a un diccionario
    row = {
        "iteration": iteration,
        "seed": seed,
        "sampling": sampling
    }
    for metric_name in rename_map.values():
        row[f"Train_{metric_name}"] = round(train_renamed[metric_name], 4)
        row[f"Val_{metric_name}"] = round(val_metrics[metric_name], 4)
    
    return row

In [61]:
def store_grid_params(grid_list, best_params_list, sampling_types, iteration, seed):
    for params, sampling in zip(best_params_list, sampling_types):
        row = {
            "iteration": iteration,
            "seed": seed,
            "sampling": sampling
        }
        row.update(params)
        grid_list.append(row)

In [62]:
def main_train(df_data, repr_name, unique_seeds, use_grid=False):
    all_metrics = []
    grid_params = []
    for i, seed in enumerate(unique_seeds):
        df_train, df_val, df_train_under, df_val_under, df_train_over, df_val_over = split(df_data, seed)
        
        if use_grid:
            best_params_base = train_with_gridsearch(df_train, df_val, i, repr_name, 'base', seed)
            best_params_under = train_with_gridsearch(df_train_under, df_val_under, i, repr_name, 'undersampling', seed)
            best_params_over = train_with_gridsearch(df_train_over, df_val_over, i, repr_name, 'oversampling', seed)
            store_grid_params(grid_list=grid_params, best_params_list=[best_params_base, best_params_under, best_params_over], sampling_types=['base', 'undersampling', 'oversampling'], iteration=i, seed=seed)
        else:
            perf_base = train(df_train, df_val, i, repr_name, 'base', seed)
            perf_under = train(df_train_under, df_val_under, i, repr_name, 'undersampling', seed)
            perf_over = train(df_train_over, df_val_over, i, repr_name, 'oversampling', seed)
        
        #all_metrics.append(metrics(best_params_base, i, seed, 'base'))
        #all_metrics.append(metrics(best_params_under, i, seed, 'undersampling'))
        #all_metrics.append(metrics(best_params_over, i, seed, 'oversampling'))

    df_metrics = pd.DataFrame(all_metrics)
    df_metrics.to_csv(f"../../models/metrics_{repr_name}_RandomForest.csv", index=False)
    if use_grid:
        df_grid_params = pd.DataFrame(grid_params)
        df_grid_params.to_csv(f"../../models/grid_params_{repr_name}_RandomForest.csv", index=False)

In [63]:
repr_name="ProtT5"
df_data = pd.read_csv(f"../../data/numerical_rep/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)

In [64]:
folder = "../../data/numerical_rep/"
unique_seeds= [42]
#unique_seeds = np.random.choice(range(100), size=30, replace=False)
#unique_seeds = [94, 42, 98, 43, 90, 44, 99, 93, 66, 34, 72, 60, 6, 39, 26, 74, 17,8, 51, 96, 53, 13, 20, 33, 29, 65, 46, 82, 79, 89]

In [65]:
print(f"Processing {repr_name}")
metrics_path = f"../../models/metrics_{repr_name}.csv"
seeds_used = unique_seeds
main_train(df_data, repr_name, seeds_used, use_grid=True)
print(f"Finished processing {repr_name}")
print("=====================================")

Processing ProtT5
GridSearchCV for Random Forest, iteration: 0
Best GridSearch params iteration 0: {'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
GridSearchCV for Random Forest, iteration: 0
Best GridSearch params iteration 0: {'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
GridSearchCV for Random Forest, iteration: 0
Best GridSearch params iteration 0: {'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Finished processing ProtT5
